# 🎨 Módulo 6: El Artesano del Escritorio — Hyprland Ricing
## Notebook de Conocimiento Guiado

**Objetivo:** Entender los conceptos técnicos detrás del ricing de escritorio Linux:
nine-slice scaling, física de cuerdas y comunicación IPC entre procesos.

**Lore:** Eres el Artesano del Escritorio Galáctico. Tu misión: crear el entorno
visual más bello y funcional del universo conocido.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | Wayland, Hyprland, Quickshell, IPC |
| 🔨 Demo | Nine-slice scaling en Python |
| 🔨 Demo | Física de cuerdas (Hooke) |
| ✏️ Ejercicio 1 | Escalar widget con nine-slice |
| ✏️ Ejercicio 2 | Simulador de cuerda simplificado |
| 🎯 Quiz | Wayland, compositing, IPC |

## 📚 Parte 1: El Stack del Ricing

### Wayland vs X11

| | X11 | Wayland |
|-|-----|---------|
| Año | 1984 | 2008 |
| Arquitectura | Servidor centralizado | Protocolo directo |
| Seguridad | Baja (cualquier app puede capturar la pantalla) | Alta (aislamiento entre apps) |
| Rendimiento | Latencia extra por servidor X | Directo al compositor |

### Hyprland
Es un **compositor Wayland** dinámico con tiles. El compositor:
1. Recibe eventos de input (teclado, ratón)
2. Gestiona las ventanas (posición, tamaño, animaciones)
3. Renderiza todo en pantalla usando OpenGL/Vulkan

### Quickshell
Framework QML para crear **widgets del sistema** (barra de estado, lanzador, etc.)
Se comunica con Hyprland vía **IPC** (sockets Unix).

### IPC (Inter-Process Communication)
```
Quickshell Widget  ──socket──►  Hyprland Compositor
                   ◄──eventos──
```
Hyprland expone un socket Unix en `/tmp/hypr/{session}/`.hyprland.sock`

In [ ]:
# 🔍 Simulación de IPC con sockets locales (sin Hyprland real)
import socket
import threading
import json
import time

class HyprlandSimulado:
    """Simula el servidor IPC de Hyprland en memoria."""
    def __init__(self):
        self.workspaces = {1: {"name": "1", "windows": 2},
                          2: {"name": "2", "windows": 0},
                          3: {"name": "3", "windows": 1}}
        self.workspace_activo = 1
    
    def ejecutar_comando(self, cmd):
        partes = cmd.strip().split()
        if not partes: return {"error": "comando vacío"}
        
        if partes[0] == "dispatch" and len(partes) >= 3 and partes[1] == "workspace":
            n = int(partes[2])
            self.workspace_activo = n
            return {"ok": True, "workspace": n}
        
        if partes[0] == "j/workspaces":
            return list(self.workspaces.values())
        
        if partes[0] == "j/activeworkspace":
            return self.workspaces.get(self.workspace_activo, {})
        
        return {"error": f"comando desconocido: {partes[0]}"}

hypr = HyprlandSimulado()

# Simular cliente IPC
comandos = [
    "j/activeworkspace",
    "j/workspaces",
    "dispatch workspace 2",
    "j/activeworkspace",
]

print("Simulación de IPC con Hyprland:")
for cmd in comandos:
    resultado = hypr.ejecutar_comando(cmd)
    print(f"  $ hyprctl {cmd}")
    print(f"    → {json.dumps(resultado)}")
    print()

## 🏗️ Parte 2: Nine-Slice Scaling

El **nine-slice** (o 9-patch) es una técnica para escalar imágenes/widgets
preservando las esquinas y bordes sin distorsión.

```
┌───┬─────────┬───┐
│ TL│  TOP    │ TR│  ← esquinas y bordes no se escalan
├───┼─────────┼───┤
│ L │ CENTER  │ R │  ← solo el centro se escala
├───┼─────────┼───┤
│ BL│ BOTTOM  │ BR│
└───┴─────────┴───┘
```

**Caso de uso:** Botones, ventanas, marcos decorativos — 
si escaláramos todo uniformemente, las esquinas se verían deformadas.

In [ ]:
# 🔨 IMPLEMENTACIÓN: Nine-Slice Scaling

def nine_slice_scale(imagen, nuevo_ancho, nuevo_alto, margen=1):
    """
    Escala una imagen 2D (lista de listas de chars) usando nine-slice.
    
    Args:
        imagen: list[list[char]] — imagen original
        nuevo_ancho, nuevo_alto: int — dimensiones objetivo
        margen: int — tamaño de las esquinas/bordes fijos
    
    Returns:
        list[list[char]] — imagen escalada
    """
    alto_orig = len(imagen)
    ancho_orig = len(imagen[0]) if imagen else 0
    
    # Validación mínima
    min_dim = 2 * margen + 1
    assert nuevo_ancho >= min_dim and nuevo_alto >= min_dim
    assert ancho_orig >= min_dim and alto_orig >= min_dim
    
    # Ancho/alto del centro escalable
    centro_src_ancho = ancho_orig - 2 * margen
    centro_src_alto  = alto_orig  - 2 * margen
    centro_dst_ancho = nuevo_ancho - 2 * margen
    centro_dst_alto  = nuevo_alto  - 2 * margen
    
    resultado = [[' '] * nuevo_ancho for _ in range(nuevo_alto)]
    
    for dst_y in range(nuevo_alto):
        for dst_x in range(nuevo_ancho):
            # Determinar de qué región viene este píxel
            if dst_x < margen:
                src_x = dst_x
            elif dst_x >= nuevo_ancho - margen:
                src_x = ancho_orig - (nuevo_ancho - dst_x)
            else:
                # Centro: escalar proporcionalmente
                t = (dst_x - margen) / max(centro_dst_ancho - 1, 1)
                src_x = margen + int(t * (centro_src_ancho - 1))
            
            if dst_y < margen:
                src_y = dst_y
            elif dst_y >= nuevo_alto - margen:
                src_y = alto_orig - (nuevo_alto - dst_y)
            else:
                t = (dst_y - margen) / max(centro_dst_alto - 1, 1)
                src_y = margen + int(t * (centro_src_alto - 1))
            
            src_x = max(0, min(src_x, ancho_orig - 1))
            src_y = max(0, min(src_y, alto_orig - 1))
            resultado[dst_y][dst_x] = imagen[src_y][src_x]
    
    return resultado

# Demo: escalar un botón ASCII
boton_original = [
    list("╔═══╗"),
    list("║ X ║"),
    list("╚═══╝"),
]

boton_grande = nine_slice_scale(boton_original, nuevo_ancho=9, nuevo_alto=5, margen=1)

print("Original (5×3):")
for fila in boton_original:
    print("  " + "".join(fila))
print()
print("Escalado (9×5) — esquinas intactas:")
for fila in boton_grande:
    print("  " + "".join(fila))

## 🏗️ Parte 3: Física de Cuerdas (Rope Physics)

La cuerda se modela como una cadena de partículas conectadas por **resortes de Hooke**:

```
F = -k × (distancia_actual - distancia_reposo)
```

En cada paso de tiempo:
1. Calcular fuerzas de resorte entre partículas vecinas
2. Aplicar gravedad
3. Actualizar velocidades y posiciones

In [ ]:
# 🔨 IMPLEMENTACIÓN: Simulador de Cuerda

import math

class Particula:
    def __init__(self, x, y, fija=False):
        self.x, self.y = float(x), float(y)
        self.vx, self.vy = 0.0, 0.0
        self.fija = fija   # Las partículas fijas no se mueven

class CuerdaSimulador:
    def __init__(self, n_particulas=6, longitud_reposo=1.0, rigidez=5.0, gravedad=9.8):
        self.particulas = [
            Particula(i * longitud_reposo, 0.0, fija=(i == 0))
            for i in range(n_particulas)
        ]
        self.longitud_reposo = longitud_reposo
        self.rigidez = rigidez
        self.gravedad = gravedad
    
    def paso(self, dt=0.05):
        """Simula un paso de tiempo dt segundos."""
        n = len(self.particulas)
        fuerzas = [(0.0, 0.0)] * n
        
        # Fuerzas de resorte entre partículas vecinas
        for i in range(n - 1):
            p1, p2 = self.particulas[i], self.particulas[i + 1]
            dx, dy = p2.x - p1.x, p2.y - p1.y
            dist = math.sqrt(dx*dx + dy*dy) or 0.0001
            fuerza = self.rigidez * (dist - self.longitud_reposo)
            fx = fuerza * dx / dist
            fy = fuerza * dy / dist
            fuerzas[i]   = (fuerzas[i][0] + fx,   fuerzas[i][1] + fy)
            fuerzas[i+1] = (fuerzas[i+1][0] - fx, fuerzas[i+1][1] - fy)
        
        # Actualizar velocidades y posiciones
        for i, p in enumerate(self.particulas):
            if p.fija: continue
            fx, fy = fuerzas[i]
            p.vx += (fx * dt) * 0.95   # amortiguación
            p.vy += (fy + self.gravedad) * dt * 0.95
            p.x += p.vx * dt
            p.y += p.vy * dt
    
    def render_ascii(self, ancho=40, alto=10):
        """Dibuja la cuerda en ASCII."""
        canvas = [[' '] * ancho for _ in range(alto)]
        for p in self.particulas:
            px = int(p.x * 4) % ancho
            py = int(p.y * 2) % alto
            if 0 <= px < ancho and 0 <= py < alto:
                canvas[py][px] = '●' if not p.fija else '■'
        return [''.join(fila) for fila in canvas]

cuerda = CuerdaSimulador(n_particulas=8)
print("Estado inicial de la cuerda:")
for fila in cuerda.render_ascii():
    if fila.strip():
        print("  |" + fila + "|")

for _ in range(30):
    cuerda.paso(dt=0.05)

print("\nDespués de 30 pasos de simulación (gravedad hacia abajo):")
posiciones = [(f"P{i}", f"({p.x:.2f}, {p.y:.2f})") for i, p in enumerate(cuerda.particulas)]
for nombre, pos in posiciones:
    print(f"  {nombre}: {pos}")

## ✏️ Ejercicio 1: Escala un widget a múltiples tamaños

In [ ]:
# ✏️ EJERCICIO 1 — Nine-slice a múltiples tamaños

ventana = [
    list("╔══════╗"),
    list("║      ║"),
    list("║      ║"),
    list("╚══════╝"),
]

# TODO: Usa nine_slice_scale para crear versiones de 12x6 y 16x8 de este widget
# Muestra los tres tamaños uno debajo del otro

# Tu código aquí...
print("TODO: Implementar escalado a múltiples tamaños")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

print("Original (8×4):")
for fila in ventana:
    print("  " + "".join(fila))

for nuevo_ancho, nuevo_alto in [(12, 6), (16, 8)]:
    escalado = nine_slice_scale(ventana, nuevo_ancho, nuevo_alto, margen=1)
    print(f"\nEscalado ({nuevo_ancho}×{nuevo_alto}):")
    for fila in escalado:
        print("  " + "".join(fila))

## ✏️ Ejercicio 2: Extiende el simulador con amortiguación variable

In [ ]:
# ✏️ EJERCICIO 2 — Cuerda con tensión ajustable

# TODO: Crea dos cuerdas, una con rigidez=2.0 y otra con rigidez=20.0
# Simula 50 pasos y compara la posición Y de la partícula final

# Tu código aquí...
print("TODO: Comparar cuerdas con diferentes rigideces")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

for rigidez in [2.0, 20.0]:
    c = CuerdaSimulador(n_particulas=6, rigidez=rigidez, gravedad=9.8)
    for _ in range(50):
        c.paso(dt=0.05)
    ultima = c.particulas[-1]
    print(f"Rigidez {rigidez:5.1f} → última partícula: ({ultima.x:.2f}, {ultima.y:.2f})")
    print(f"  {'Cuerda rígida: se dobla menos' if rigidez > 10 else 'Cuerda floja: cae más'}")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Por qué Wayland no puede capturar ventanas de otras apps fácilmente?
> **R:** En Wayland, cada ventana pertenece a su propio surface y el compositor
> controla el buffer. No hay servidor centralizado que todas las apps compartan,
> por lo que una app no puede acceder directamente al buffer de otra.

**P2:** ¿Qué es un compositor de pantalla y cómo funciona?
> **R:** Recibe buffers de cada ventana (DMA-BUF), los combina con OpenGL/Vulkan
> y los envía al framebuffer del monitor. Gestiona transparencias, animaciones y VSync.

**P3:** ¿Por qué usaríamos nine-slice en lugar de escalar toda la imagen?
> **R:** Escalar uniformemente distorsiona las esquinas y bordes (ej: bordes redondeados
> se volverían elipses). Nine-slice preserva las esquinas y estira solo el centro.

**P4:** ¿Qué ventaja tiene el modelo de física de resortes sobre keyframes para animaciones?
> **R:** Las animaciones basadas en física son "springy" y se adaptan automáticamente
> a cambios de posición sin definir curvas manualmente. Son más naturales y el código
> es más corto (solo parámetros: rigidez, amortiguación).